# Atividade Prática — Aula 3: Limpeza, Preparação e Qualidade dos Dados com Pandas

Esta atividade foi construída com base nos slides da Aula 3, que destacam a **limpeza como a fundação invisível** de qualquer dashboard confiável. O objetivo aqui não é apenas "fazer funcionar", mas tomar decisões conscientes sobre tipos, valores ausentes, duplicidades, variáveis derivadas e exportação da base limpa. 

## Regras desta atividade
- Você deve **construir os códigos**.
- O notebook orienta os passos, mas não entrega as soluções prontas.
- Sempre que fizer uma decisão de limpeza, **documente o porquê** em comentário ou célula markdown.
- Ao final, exporte uma base limpa para uso nas próximas aulas.

## Dataset desta atividade
Arquivo bruto: `vendas_brasil_aula3_bruto.csv`


## 1. Preparação do ambiente

Importe as bibliotecas necessárias para trabalhar com:
- leitura de dados
- tratamento de nulos
- identificação de infinitos
- exportação do resultado final

**Sugestão:** Pandas e NumPy já resolvem toda a atividade.


In [ ]:
import os
import pandas as pd
import numpy as np

## 2. Leitura da base bruta

Leia o arquivo `vendas_brasil_aula3_bruto.csv` em um DataFrame chamado `df`.

Depois:
1. Exiba as primeiras linhas
2. Exiba as últimas linhas
3. Observe visualmente se já existem sinais de sujeira


In [ ]:
CSV_PATHS = [
    'data/vendas_brasil_aula3_bruto.csv'
]

csv_path = next((p for p in CSV_PATHS if os.path.exists(p)), None)
df = pd.read_csv(csv_path)

df.head() #Primeiras linhas
df.tail() #Ultimas linhas


## 3. Check-up inicial do dataset

Com base no checklist de um dataset "clean" apresentado na aula, faça um diagnóstico inicial da base. 

### Sua missão
Use comandos que permitam responder:
- Qual é o tamanho da base?
- Quais são os tipos atuais das colunas?
- Existem valores nulos?
- Há colunas com tipo inadequado?
- Há algo que pareça impedir análises confiáveis?


In [ ]:
print(f"O dataset possui {df.shape[0]} linhas e {df.shape[1]} colunas.\n") # Tamanho 
df.info() # Tipos de colunas

print(f"Valores nulos por coluna->\n{df.isnull().sum()}")


print(df['uf'].unique())
df['pedido_id'].duplicated().any()
# --- COLUNAS COM TIPO INADEQUADO ---
# 'data': formatos mistos (2024/01/01 e 2024-01-01).
# 'receita': valores com "R$" e vírgulas (ex: R$ 748,80).
# 'uf': sujeira de digitação (ex: 'SP', 'rj', ' mg').

# --- IMPEDIMENTOS PARA ANÁLISES CONFIÁVEIS ---
# 1. Duplicatas: Existem 8 IDs de pedidos repetidos (infla o resultado).
# 2. Inconsistência: 'MG', 'mg' e ' mg' seriam contados como 3 estados diferentes.
# 3. Erros: Vendas com receita R$ 0,00 que possuem lucro.

## 4. Batalha 1 — A tirania dos tipos de dados

Nos slides, vimos que datas não podem ser tratadas como texto e que números em formato string quebram análises. 
### Tarefa
Converta corretamente, quando fizer sentido:
- `data`
- `receita`
- `lucro`
- `quantidade`

### Orientação
- Investigue valores estranhos antes de converter
- Pense no uso de `errors='coerce'`
- Registre em markdown ou comentário quais problemas encontrou


In [ ]:
# Sujeiras
df[df['receita'].astype(str).str.contains('[a-zA-Z]', na=False)]['receita'].unique()


# --- 1. RECEITA ---
# Problemas: "R$", pontos como milhar e vírgula como decimal (ex: R$ 1.200,50)
df['receita'] = df['receita'].astype(str).str.replace('R$', '', regex=False).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
df['receita'] = pd.to_numeric(df['receita'], errors='coerce')

# --- 2. DATA ---
# Problema: Formatos mistos (2024/01/01 e 2024-01-01)
# O format='mixed' ou dayfirst=False ajuda o pandas a entender a bagunça
df['data'] = pd.to_datetime(df['data'], errors='coerce')

# --- 3. LUCRO E QUANTIDADE ---
# Garantindo que sejam numéricos (coerce transforma o que for texto/lixo em NaN)
df['lucro'] = pd.to_numeric(df['lucro'], errors='coerce')
df['quantidade'] = pd.to_numeric(df['quantidade'], errors='coerce')



### Reflexão curta
Explique:
1. Por que deixar `data` como texto pode quebrar análises temporais?
2. Por que deixar `receita` como string pode distorcer cálculos?

R:
A manutenção de data e receita como texto inviabiliza a análise pois o computador deixa de interpretar os valores logicamente e passa a tratá-los apenas como símbolos

Para datas, isso destrói a cronologia ao aplicar uma ordenação alfabética, impedindo cálculos de prazos e sazonalidade; para a receita, a tipagem como string impossibilita qualquer operação matemática e causa erros de "concatenação", gerando relatórios financeiros falsos e sem valor estratégico.

## 5. Batalha 2 — O enigma dos valores ausentes

A aula reforça que valores ausentes não devem ser tratados automaticamente; a decisão precisa ser justificada. fileciteturn2file0

### Tarefa
1. Mapeie os valores ausentes por coluna
2. Identifique quais colunas críticas têm nulos
3. Defina uma estratégia para cada caso:
   - remover linhas?
   - preencher?
   - manter?
4. Justifique cada escolha

### Dica
Evite decisões automáticas sem análise de contexto.


In [ ]:
# 1. Tratando as colunas categóricas para não perder a venda
cols_categoricas = ['uf', 'canal', 'categoria']
df[cols_categoricas] = df[cols_categoricas].fillna('Não Informado')

# 2. Removendo linhas onde a Receita é nula (essencial para o financeiro)
df.dropna(subset=['receita'], inplace=True)

# 3. Tratando o Lucro (decisão conservadora: usar a mediana para evitar distorção por outliers)
mediana_lucro = df['lucro'].median()
df['lucro'] = df['lucro'].fillna(mediana_lucro)

# 4. 
"""
Financeiro (receita): REMOVER. Se não tem valor de venda, o dado é "lixo" para o faturamento.

Categorias (uf, canal, categoria): PREENCHER. Use "Não Informado". Isso evita que você perca a linha inteira e permite que o dinheiro dessas vendas ainda seja somado no total geral.

Performance (lucro): MEDIANA. Preencher com o valor central da coluna. É a forma mais segura de manter a linha sem distorcer a média por causa de valores muito altos ou baixos.

Texto (observacao): MANTER. Nulo aqui não quebra conta nenhuma. Apenas ignore.
"""

### Registro reflexivo
Escreva aqui sua justificativa para o tratamento dos nulos:
- Em quais colunas você removeu linhas?
- Em quais colunas você preencheu valores?
- Em quais situações decidiu manter o nulo?


## 6. Batalha 3 — O ataque dos clones (duplicidades)

A aula alerta que nem toda duplicidade é erro automático: pode haver compra legítima repetida ou dupla inserção do sistema. fileciteturn2file0

### Tarefa
1. Investigue se há linhas duplicadas
2. Analise se a duplicidade parece nociva para os KPIs
3. Escolha uma estratégia:
   - remover duplicidades totais?
   - remover apenas com base em certas colunas?
   - manter?
4. Justifique a decisão


In [ ]:
# --- INVESTIGAÇÃO E ESTRATÉGIA DE DUPLICIDADES ---

# 1. Identificação: Foram detectados 8 IDs de pedido duplicados (pedido_id).
# 2. Impacto: Manter IDs repetidos inflaria o faturamento e a contagem de vendas, gerando KPIs falsos.
# 3. Decisão: Remover as duplicidades mantendo apenas a primeira ocorrência de cada pedido_id.
# 4. Justificativa: Em sistemas de vendas, o pedido_id é uma chave exclusiva; repetições indicam erro de 
# processamento ou exportação (dupla inserção), e não novas transações legítimas do mesmo cliente.

df.drop_duplicates(subset=['pedido_id'], keep='first', inplace=True)



## 7. Padronização de categorias

Os slides mostram que categorias duplicadas ou mal escritas podem distorcer rankings e filtros. 

### Tarefa
Inspecione e padronize, se necessário:
- `uf`
- `canal`
- `categoria`

### Pense em:
- maiúsculas e minúsculas
- espaços extras
- acentuação / variações
- categorias equivalentes escritas de formas diferentes


In [ ]:


# 1. Investigação: Identificamos UFs com erros de caixa (RJ vs rj) e espaços extras (' mg', 'BA ').
# 2. Impacto: Sem padronização, o agrupamento por estado ou canal ficaria fragmentado (ex: 3 barras para MG).
# 3. Estratégia: Aplicar .str.upper() para uniformizar a caixa e .str.strip() para remover espaços invisíveis.

for col in ['uf', 'canal', 'categoria']:
    df[col] = df[col].astype(str).str.upper().str.strip()

# 4. Ajuste de equivalentes: Garantir que variações específicas sejam corrigidas (ex: 'MG' e 'MINAS' se houver)


## 8. Subindo de nível — Criação de variáveis derivadas

Depois da limpeza, é hora de enriquecer a base com variáveis úteis para análise. 

### Tarefa
Crie, se possível:
- `ano`
- `mes`
- `ano_mes`
- `margem_lucro`

### Atenção
A criação de `margem_lucro` exige cuidado com divisões por zero e possíveis valores infinitos.


In [ ]:
# 1. Temporais: Extração de grão para análises de sazonalidade e tendência
df['ano'] = df['data'].dt.year
df['mes'] = df['data'].dt.month
df['ano_mes'] = df['data'].dt.to_period('M') # Formato YYYY-MM para agrupamentos mensais

# 2. Financeira: Cálculo da rentabilidade (Margem de Lucro)
# Justificativa: Usamos np.where para evitar erro de divisão por zero onde receita é 0 ou NaN.
import numpy as np
df['margem_lucro'] = np.where(df['receita'] > 0, (df['lucro'] / df['receita']), 0)



### Reflexão técnica
Explique:
1. O que pode acontecer ao calcular margem de lucro quando a receita é zero?
2. Como você decidiu tratar esse caso?

O Problema: Tentar dividir o lucro por uma receita zero gera um erro de 
 "ZeroDivisionError" ou resulta em valores infinitos no Pandas. 
 Matematicamente, a margem é indefinida quando não há venda.

Decisão e Solução: Usei a função np.where() para criar uma trava lógica. 
 Se a receita for maior que zero, o cálculo (lucro / receita) é executado; 
 caso contrário, atribuímos 0. Isso mantém a coluna numérica e evita 
 que valores "inf" quebrem as médias e os gráficos do dashboard.



## 9. Seleção final — Menos é mais

A aula propõe que nem toda coluna precisa seguir para a base analítica final. 

### Tarefa
Crie um DataFrame final, por exemplo `df_clean` ou `df_dash`, contendo apenas as colunas estritamente necessárias para análises de negócio.

**Sugestão de raciocínio:**
- Quais colunas ajudam a responder perguntas de negócio?
- Quais colunas são operacionais, auxiliares ou dispensáveis?
- O que precisa existir para as próximas aulas de visualização e dashboard?


In [ ]:
# 1. Colunas Mantidas: data, ano, mes, ano_mes, uf, canal, categoria, produto, quantidade, receita, lucro, margem_lucro.
#    Justificativa: Estas são as "dimensões" e "fatos" necessários para responder perguntas de negócio como 
#    "Qual categoria lucra mais?" ou "Como as vendas evoluem mês a mês?".

# 2. Colunas Removidas: pedido_id, cliente_id, observacao.
#    Justificativa: 'pedido_id' e 'cliente_id' são chaves operacionais úteis para rastreio no banco, mas não para 
#    estatísticas. 'observacao' é um campo de texto livre que apenas ocuparia memória sem agregar valor ao dashboard.

colunas_analiticas = [
    'data', 'ano', 'mes', 'ano_mes', 'uf', 'canal', 
    'categoria', 'produto', 'quantidade', 'receita', 
    'lucro', 'margem_lucro'
]

# Criando o DataFrame final otimizado para a próxima etapa de visualização
df_clean = df[colunas_analiticas].copy()

# O df_clean agora contém apenas dados estruturados, limpos e enriquecidos para o dashboard.
display(df_clean.head())

## 10. Validação final da base limpa

Antes de exportar, faça uma checagem final.

### Verifique:
- os tipos estão corretos?
- ainda há nulos problemáticos?
- ainda há duplicidades nocivas?
- existe algum infinito em `margem_lucro`?
- a base está pronta para ser usada em análises e dashboards?


In [ ]:
# 1. Tipos e Estrutura: Verificando se as conversões de data e números foram mantidas.
# 2. Nulos e Infinitos: Garantindo que não existam valores que quebrem cálculos (como Inf na margem).
# 3. Integridade: Confirmando que a limpeza de duplicatas e padronização de texto funcionou.

print("--- CHECKLIST DE VALIDAÇÃO ---")

# Verificação de Tipos e Nulos
print(df_clean.info())

# Verificação de Infinitos na Margem de Lucro (ocorre se houver divisão por zero não tratada)
print(f"\nInfinitos em margem_lucro: {np.isinf(df_clean['margem_lucro']).sum()}")

# Verificação de Nulos Restantes
print(f"Nulos restantes:\n{df_clean.isnull().sum()}")

# Validação Visual da Padronização
print(f"\nUFs únicas: {df_clean['uf'].unique()}")


## 11. Exportação da base "Clean"

Agora gere o arquivo final da base tratada.

### Tarefa
Exporte sua base limpa com o nome:

`vendas_brasil_aula3_clean.csv`

### Observação
Esse arquivo será o selo de qualidade do seu trabalho desta aula.


In [ ]:
try:
    df_clean.to_csv('vendas_brasil_aula3_clean.csv', index=False, encoding='utf-8-sig')
    print("Sucesso: Arquivo 'vendas_brasil_aula3_clean.csv' gerado com sucesso.")
except Exception as e:
    print(f"Erro ao exportar: {e}")

# Check final de sanidade do arquivo gerado
print(f"Linhas exportadas: {len(df_clean)}")
print(f"Colunas exportadas: {list(df_clean.columns)}")

## 12. Conclusão e registro reflexivo

Escreva um pequeno texto respondendo:

1. Quais foram os principais problemas de qualidade encontrados?
2. Qual decisão de limpeza foi mais difícil?
3. Que impacto essas falhas poderiam causar em um dashboard?
4. Por que a etapa de limpeza é considerada a "fundação invisível" do projeto?



Os principais problemas de qualidade encontrados nesta base bruta foram a inconsistência de tipos (datas e valores financeiros lidos como texto), a presença de duplicidades de IDs de pedidos e a falta de padronização em categorias como UF, onde erros de digitação e espaços extras fragmentavam os dados. A decisão de limpeza mais difícil foi o manejo dos valores nulos, especialmente na coluna de receita e lucro, pois exige um equilíbrio entre descartar dados corrompidos para manter a precisão financeira ou tentar recuperá-los para não reduzir drasticamente o volume da amostra.

O impacto destas falhas num dashboard seria desastroso: as datas como texto impediriam qualquer ordenação cronológica correta, as duplicidades inflariam o faturamento total e as categorias não padronizadas gerariam gráficos com barras duplicadas para a mesma região ou produto. Por fim, a limpeza é a fundação invisível porque o sucesso de qualquer análise depende da confiança nos dados; se a base estiver "suja", o dashboard entregará conclusões erradas, provando que nenhum visual sofisticado consegue compensar um dado tecnicamente mal tratado.

## 13. Desafio extra (opcional)

Use sua base limpa para responder rapidamente:

- Qual UF concentra maior receita?
- Qual canal gera maior lucro?
- Existe algum mês com desempenho claramente acima dos demais?

Você pode responder com tabelas simples ou pequenos agrupamentos.


In [ ]:
# Desafio extra opcional
